<a href="https://colab.research.google.com/github/sw030701-ai/motor-control-optimization/blob/main/experiments/03_pid_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · PID Optimization Experiment
### Manual Baseline PID → Random Search → Bayesian Optimization → Fair Comparison

---

### Objective

이 notebook의 목적은 앞에서 만든 **Manual Baseline PID**를 benchmark로 두고, 동일한 nominal motor / reference / simulation condition에서 PID gains를 자동 최적화한 뒤 성능을 비교하는 것이다.

```text
Manual Baseline PID
      ↓
Same plant + same reference + same simulation settings
      ↓
Random Search PID Optimization
      ↓
Bayesian Optimization PID Optimization
      ↓
Cost + dynamic response metrics 비교
```

중요한 점은 baseline PID가 **cost-minimized controller가 아니라 response-based acceptance rule로 정한 conventional benchmark**라는 것이다. Optimization은 같은 조건에서 cost function `J`를 줄이는 gains를 찾고, 최종 평가는 `J`뿐 아니라 `M_p`, settling time, steady-state error, control effort, saturation까지 함께 본다.


In [1]:
import os, sys, json, platform, subprocess, math, warnings
from pathlib import Path


def _in_colab():
    return "google.colab" in sys.modules


def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").exists() and (cand / "docs").exists():
            return cand
    return p


REPO_URL = "https://github.com/sw030701-ai/motor-control-optimization.git"

if _in_colab():
    root = Path("/content/motor-control-optimization")
    if not root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(root)], check=True)
    else:
        subprocess.run(["git", "pull", "--ff-only"], cwd=root, check=False)
    ROOT = root
else:
    ROOT = _find_root(Path.cwd())

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/xdgcache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
if not _in_colab():
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False


def _git(*args):
    try:
        return subprocess.check_output(["git", *args], cwd=ROOT, text=True).strip()
    except Exception:
        return None

ENV = {
    "root": str(ROOT),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "git_commit": _git("rev-parse", "HEAD"),
    "git_dirty": bool(_git("status", "--short")),
}

print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "root": "/Users/seong-uuimac/Documents/Codex/2026-09-05/referenced-chatgpt-conversation-this-is-an-2/work/motor-control-optimization",
  "python": "3.13.9",
  "numpy": "2.3.5",
  "pandas": "2.3.3",
  "git_commit": "3625b508836bdfd3dd94b9486a461103697b5bb2",
  "git_dirty": true
}


In [2]:
SAVE_ARTIFACTS = True
RANDOM_SEED = 42

RESULT_TABLE_DIR = Path("results") / "tables"
RESULT_FIGURE_DIR = Path("results") / "figures"
if SAVE_ARTIFACTS:
    RESULT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
    RESULT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("SAVE_ARTIFACTS:", SAVE_ARTIFACTS)
print("RANDOM_SEED:", RANDOM_SEED)


SAVE_ARTIFACTS: True
RANDOM_SEED: 42


## Section 1 — Load Project Modules / Config

이 section에서는 docs와 이전 experiment output에 이미 고정된 값을 불러온다.

- Nominal motor parameters: `docs/theory/motor_model.md`, `src.motor.dc_motor.nominal_dc_motor_params()`
- Reference speed, simulation time, dt: `results/tables/reference_selection_record.json`
- Baseline PID gains / baseline metrics: `results/tables/baseline_pid_tuning_record.json`
- Cost weights: `docs/theory/cost_function.md`, `src.optimization.cost_function.compute_cost()` default
- Acceptance criteria: `docs/experiments/baseline_pid_tuning.md`, `src.optimization.cost_function.accepted_baseline()`

Optimization 중 optimizer가 바꾸는 것은 오직 `K_p`, `K_i`, `K_d`이다.


In [3]:
from src.controller.pid import PIDGains
from src.motor.dc_motor import NOMINAL_MOTOR_SOURCE, nominal_dc_motor_params
from src.optimization.cost_function import accepted_baseline, compute_cost
from src.optimization.pid_optimization import (
    PIDOptimizationConfig,
    evaluate_pid_candidate,
    random_search_pid,
)
from src.simulation.pid_simulation import reference_from_reachable_speed, simulate_pid

params = nominal_dc_motor_params()
V_MAX = 12.0

REFERENCE_FILE = RESULT_TABLE_DIR / "reference_selection_record.json"
BASELINE_FILE = RESULT_TABLE_DIR / "baseline_pid_tuning_record.json"

if REFERENCE_FILE.exists():
    reference_record = json.loads(REFERENCE_FILE.read_text(encoding="utf-8"))
else:
    reference_record = {
        "V_max": V_MAX,
        "omega_ss_max_rad_s": params.no_load_steady_state_speed(V_MAX),
        "reference_fraction": 0.50,
        "omega_ref_rad_s": round(reference_from_reachable_speed(params, V_MAX, fraction=0.50), 2),
        "simulation_time_s": 10.0,
        "dt_s": 0.001,
        "note": "Recomputed because 01 reference record was not found.",
    }

if BASELINE_FILE.exists():
    baseline_source_record = json.loads(BASELINE_FILE.read_text(encoding="utf-8"))
else:
    baseline_source_record = {
        "K_p_baseline": 0.8,
        "K_i_baseline": 2.0,
        "K_d_baseline": 0.002,
        "note": "Fallback from docs/experiments/baseline_pid_tuning.md because baseline result file was not found.",
    }

OMEGA_REF = float(reference_record["omega_ref_rad_s"])
SIMULATION_TIME = float(reference_record.get("simulation_time_s", 10.0))
DT = float(reference_record.get("dt_s", 0.001))
LOAD_TORQUE = 0.0

COST_WEIGHTS = {
    "w_tracking": 0.60,
    "w_overshoot": 0.25,
    "w_control": 0.15,
}

ACCEPTANCE = {
    "overshoot_percent_max": 10.0,
    "steady_state_error_percent_max": 2.0,
    "saturation_percent_max_from_code": 5.0,
    "sustained_oscillation": "avoid",
}

fixed_condition_summary = pd.DataFrame([{
    "R": params.R,
    "L": params.L,
    "J_m": params.J_m,
    "b": params.b,
    "K_t": params.K_t,
    "K_e": params.K_e,
    "omega_ref_rad_s": OMEGA_REF,
    "V_max": V_MAX,
    "simulation_time_s": SIMULATION_TIME,
    "dt_s": DT,
    "initial_current_A": 0.0,
    "initial_speed_rad_s": 0.0,
    "load_torque_Nm": LOAD_TORQUE,
}])

display(fixed_condition_summary.T.rename(columns={0: "value"}))
print("Reference record:")
print(json.dumps(reference_record, indent=2, ensure_ascii=False))
print("\nCost weights:")
print(json.dumps(COST_WEIGHTS, indent=2, ensure_ascii=False))
print("\nAcceptance criteria:")
print(json.dumps(ACCEPTANCE, indent=2, ensure_ascii=False))


,value
R,0.186440
L,0.006300
J_m,0.013767
b,0.049813
K_t,0.020375
K_e,0.020375
omega_ref_rad_s,12.600000
V_max,12.000000
simulation_time_s,10.000000
dt_s,0.001000


Reference record:
{
  "V_max": 12.0,
  "omega_ss_max_rad_s": 25.200271699744086,
  "reference_fraction": 0.5,
  "omega_ref_rad_s": 12.6,
  "simulation_time_s": 10.0,
  "dt_s": 0.001
}

Cost weights:
{
  "w_tracking": 0.6,
  "w_overshoot": 0.25,
  "w_control": 0.15
}

Acceptance criteria:
{
  "overshoot_percent_max": 10.0,
  "steady_state_error_percent_max": 2.0,
  "saturation_percent_max_from_code": 5.0,
  "sustained_oscillation": "avoid"
}


## Section 2 — Baseline Benchmark

Baseline gains는 `02_pid_baseline_tuning.ipynb`에서 response-based acceptance rule로 고정한 값이다.

```text
K_p_baseline = 0.80
K_i_baseline = 2.00
K_d_baseline = 0.002
```

이 baseline은 optimization input이 아니라, optimization 이후 비교할 benchmark이다.


In [4]:
baseline_gains = PIDGains(
    K_p=float(baseline_source_record["K_p_baseline"]),
    K_i=float(baseline_source_record["K_i_baseline"]),
    K_d=float(baseline_source_record["K_d_baseline"]),
)

config = PIDOptimizationConfig(
    omega_ref=OMEGA_REF,
    V_max=V_MAX,
    simulation_time=SIMULATION_TIME,
    dt=DT,
    load_torque=LOAD_TORQUE,
)

baseline_result = simulate_pid(
    motor_params=params,
    gains=baseline_gains,
    omega_ref=OMEGA_REF,
    V_max=V_MAX,
    simulation_time=SIMULATION_TIME,
    dt=DT,
    load_torque=LOAD_TORQUE,
)
baseline_cost = compute_cost(baseline_result, omega_ref=OMEGA_REF, V_max=V_MAX)

baseline_summary = pd.DataFrame([{
    "Controller": "Manual Baseline",
    "K_p": baseline_gains.K_p,
    "K_i": baseline_gains.K_i,
    "K_d": baseline_gains.K_d,
    "J_total": baseline_cost["total"],
    "J_tracking": baseline_cost["tracking"],
    "J_overshoot": baseline_cost["overshoot_cost"],
    "J_control": baseline_cost["control"],
    "overshoot_percent": baseline_cost["overshoot_percent"],
    "settling_time": baseline_cost["settling_time"],
    "steady_state_error_percent": baseline_cost["steady_state_error_percent"],
    "voltage_max_abs": baseline_cost["voltage_max_abs"],
    "saturation_percent": baseline_cost["saturation_percent"],
    "omega_final": baseline_cost["omega_final"],
    "accepted_v1": accepted_baseline(baseline_cost),
}])

display(baseline_summary.T.rename(columns={0: "value"}))


,value
Controller,Manual Baseline
K_p,0.8
K_i,2.0
K_d,0.002
J_total,0.038177
J_tracking,0.000114
J_overshoot,0.0
J_control,0.254054
overshoot_percent,0.0
settling_time,1.322


## Section 3 — Search Space and Evaluation Budget

`docs/experiments/optimization_plan.md`에는 gain bounds와 evaluation budget이 아직 `TBD`로 남아 있다.

따라서 이 notebook에서는 새 숫자를 docs source of truth처럼 만들지 않는다. 대신 notebook을 실행 가능한 상태로 두기 위해, 아래 값을 **temporary implementation choice**로 분리한다.

- Search bounds: `src/simulation/baseline_tuning.py`의 manual scan candidate min/max를 사용한다.
- Evaluation budget: manual baseline scan의 총 candidate 수 `10 + 7 + 6 = 23`을 각 optimizer에 동일하게 적용한다.

나중에 docs의 optimization plan에 공식 bounds와 budget이 확정되면, 이 section만 그 값으로 교체하면 된다.


In [5]:
BASELINE_SCAN_CANDIDATES = {
    "K_p": [0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50, 0.80],
    "K_i": [0.05, 0.10, 0.20, 0.40, 0.80, 1.20, 2.00],
    "K_d": [0.0, 0.0001, 0.0002, 0.0005, 0.0010, 0.0020],
}

GAIN_BOUNDS = {
    gain: (min(values), max(values))
    for gain, values in BASELINE_SCAN_CANDIDATES.items()
}

EVALUATION_BUDGET = sum(len(values) for values in BASELINE_SCAN_CANDIDATES.values())
N_RANDOM_TRIALS = EVALUATION_BUDGET
N_BAYESIAN_TRIALS = EVALUATION_BUDGET

search_summary = pd.DataFrame([
    {
        "Gain": gain,
        "Lower Bound": low,
        "Upper Bound": high,
        "Source": "baseline_tuning.py scan range; optimization_plan.md is TBD",
    }
    for gain, (low, high) in GAIN_BOUNDS.items()
])

budget_summary = pd.DataFrame([
    {"Method": "Random Search", "Evaluation Budget": N_RANDOM_TRIALS, "Source": "temporary equal budget from baseline scan count"},
    {"Method": "Bayesian Optimization", "Evaluation Budget": N_BAYESIAN_TRIALS, "Source": "temporary equal budget from baseline scan count"},
])

display(search_summary)
display(budget_summary)


,Gain,Lower Bound,Upper Bound,Source
0,K_p,0.02,0.800,baseline_tuning.py scan range; optimization_pl...
1,K_i,0.05,2.000,baseline_tuning.py scan range; optimization_pl...
2,K_d,0.00,0.002,baseline_tuning.py scan range; optimization_pl...


,Method,Evaluation Budget,Source
0,Random Search,23,temporary equal budget from baseline scan count
1,Bayesian Optimization,23,temporary equal budget from baseline scan count


## Section 4 — Random Search Optimization

Random Search는 정해진 search bounds 안에서 `K_p`, `K_i`, `K_d`를 uniform random sampling한다.

각 candidate는 같은 procedure로 평가한다.

```text
candidate gains
      ↓
closed-loop PID simulation
      ↓
compute_cost()
      ↓
trial history에 기록
```

Unstable 또는 invalid candidate는 helper에서 높은 penalty cost로 기록한다. Persistent saturation은 현재 v1 cost에 직접 penalty로 들어가 있지 않으므로, comparison metric으로 따로 확인한다.


In [6]:
random_records, random_best = random_search_pid(
    motor_params=params,
    bounds=GAIN_BOUNDS,
    n_trials=N_RANDOM_TRIALS,
    config=config,
    seed=RANDOM_SEED,
)

random_history = pd.DataFrame(random_records)
random_history["best_so_far"] = random_history["total"].cummin()

print("Random Search best candidate:")
display(pd.DataFrame([random_best]).T.rename(columns={0: "value"}))


Random Search best candidate:


,value
K_p,0.563947
K_i,0.233646
K_d,0.001951
total,0.03258
tracking,0.009169
overshoot_cost,0.0
control,0.180529
omega_final,12.036953
omega_max,12.036953
overshoot,0.0


## Section 5 — Bayesian Optimization

Bayesian Optimization은 지금까지 평가한

```text
(K_p, K_i, K_d) → J
```

관계를 Gaussian Process surrogate model로 근사하고, Expected Improvement acquisition으로 다음 candidate를 고른다.

이 repo에는 아직 optimizer package가 고정되어 있지 않으므로, Colab에 기본적으로 포함되는 경우가 많은 `scikit-learn` + `scipy`로 최소 구현한다. 이것은 docs에 확정된 final implementation이 아니라, 현재 v1 plan을 실행 가능한 형태로 만든 implementation detail이다.


In [7]:
try:
    from scipy.stats import norm
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
    SKLEARN_AVAILABLE = True
except ImportError as exc:
    SKLEARN_AVAILABLE = False
    SKLEARN_IMPORT_ERROR = exc

print("SKLEARN_AVAILABLE:", SKLEARN_AVAILABLE)
if not SKLEARN_AVAILABLE:
    print("Bayesian Optimization requires scipy and scikit-learn.")
    print("In Colab, run: %pip install scipy scikit-learn")


SKLEARN_AVAILABLE: True


In [8]:
def _bounds_arrays(bounds):
    names = ["K_p", "K_i", "K_d"]
    lows = np.array([bounds[name][0] for name in names], dtype=float)
    highs = np.array([bounds[name][1] for name in names], dtype=float)
    return names, lows, highs


def _scale(x, lows, highs):
    return (np.asarray(x, dtype=float) - lows) / (highs - lows)


def _unscale(z, lows, highs):
    return lows + np.asarray(z, dtype=float) * (highs - lows)


def bayesian_optimization_pid(
    motor_params,
    bounds,
    n_trials,
    config,
    seed=42,
    n_initial=5,
    acquisition_pool_size=1000,
):
    if not SKLEARN_AVAILABLE:
        raise ImportError("scipy and scikit-learn are required for Bayesian Optimization.")

    rng = np.random.default_rng(seed)
    names, lows, highs = _bounds_arrays(bounds)
    records = []
    x_history = []
    n_initial = min(n_initial, n_trials)

    def evaluate_array(x, trial):
        gains = PIDGains(K_p=float(x[0]), K_i=float(x[1]), K_d=float(x[2]))
        record = evaluate_pid_candidate(motor_params, gains, config)
        record["trial"] = trial
        return record

    for trial in range(1, n_initial + 1):
        x = _unscale(rng.random(len(names)), lows, highs)
        x_history.append(x)
        records.append(evaluate_array(x, trial))

    for trial in range(n_initial + 1, n_trials + 1):
        x_train = _scale(np.vstack(x_history), lows, highs)
        y_train = np.array([record["total"] for record in records], dtype=float)

        kernel = (
            ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3))
            * Matern(length_scale=np.ones(len(names)), nu=2.5)
            + WhiteKernel(noise_level=1e-8, noise_level_bounds=(1e-10, 1e-3))
        )
        model = GaussianProcessRegressor(
            kernel=kernel,
            normalize_y=True,
            random_state=seed,
            n_restarts_optimizer=2,
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(x_train, y_train)

        candidate_pool = rng.random((acquisition_pool_size, len(names)))
        mu, sigma = model.predict(candidate_pool, return_std=True)
        sigma = np.maximum(sigma, 1e-12)
        best_observed = np.min(y_train)
        improvement = best_observed - mu
        z = improvement / sigma
        expected_improvement = improvement * norm.cdf(z) + sigma * norm.pdf(z)

        x_next = _unscale(candidate_pool[int(np.argmax(expected_improvement))], lows, highs)
        x_history.append(x_next)
        records.append(evaluate_array(x_next, trial))

    best = min(records, key=lambda record: record["total"])
    return records, best


bayesian_records, bayesian_best = bayesian_optimization_pid(
    motor_params=params,
    bounds=GAIN_BOUNDS,
    n_trials=N_BAYESIAN_TRIALS,
    config=config,
    seed=RANDOM_SEED,
)

bayesian_history = pd.DataFrame(bayesian_records)
bayesian_history["best_so_far"] = bayesian_history["total"].cummin()

print("Bayesian Optimization best candidate:")
display(pd.DataFrame([bayesian_best]).T.rename(columns={0: "value"}))


Bayesian Optimization best candidate:


,value
K_p,0.387395
K_i,0.233312
K_d,0.001882
total,0.032488
tracking,0.009537
overshoot_cost,0.0
control,0.178437
omega_final,12.187956
omega_max,12.187956
overshoot,0.0


## Section 6 — Comparison Table

세 controller를 같은 motor / reference / simulation condition에서 다시 simulate하고 같은 `compute_cost()`로 평가한다.

비교할 핵심은 다음과 같다.

- `J_total`: optimizer의 objective가 얼마나 줄었는가
- `J_tracking`, `J_overshoot`, `J_control`: total cost가 왜 변했는가
- `M_p`, settling time, SSE: dynamic response가 실제로 좋아졌는가
- voltage / saturation: actuator 부담이 커졌는가 줄었는가


In [9]:
def simulate_and_summarize(label, gains):
    result = simulate_pid(
        motor_params=params,
        gains=gains,
        omega_ref=OMEGA_REF,
        V_max=V_MAX,
        simulation_time=SIMULATION_TIME,
        dt=DT,
        load_torque=LOAD_TORQUE,
    )
    cost = compute_cost(result, omega_ref=OMEGA_REF, V_max=V_MAX)
    return result, {
        "Controller": label,
        "K_p": gains.K_p,
        "K_i": gains.K_i,
        "K_d": gains.K_d,
        "J_total": cost["total"],
        "J_tracking": cost["tracking"],
        "J_overshoot": cost["overshoot_cost"],
        "J_control": cost["control"],
        "overshoot_percent": cost["overshoot_percent"],
        "settling_time": cost["settling_time"],
        "steady_state_error_percent": cost["steady_state_error_percent"],
        "voltage_max_abs": cost["voltage_max_abs"],
        "saturation_percent": cost["saturation_percent"],
        "omega_final": cost["omega_final"],
        "accepted_v1": accepted_baseline(cost),
    }

random_gains = PIDGains(
    K_p=float(random_best["K_p"]),
    K_i=float(random_best["K_i"]),
    K_d=float(random_best["K_d"]),
)
bayesian_gains = PIDGains(
    K_p=float(bayesian_best["K_p"]),
    K_i=float(bayesian_best["K_i"]),
    K_d=float(bayesian_best["K_d"]),
)

responses = {}
rows = []
for label, gains in [
    ("Manual Baseline", baseline_gains),
    ("Random Search", random_gains),
    ("Bayesian Optimization", bayesian_gains),
]:
    result, row = simulate_and_summarize(label, gains)
    responses[label] = result
    rows.append(row)

comparison = pd.DataFrame(rows)
baseline_j = float(comparison.loc[comparison["Controller"] == "Manual Baseline", "J_total"].iloc[0])
comparison["J_improvement_vs_baseline_percent"] = (
    (baseline_j - comparison["J_total"]) / baseline_j * 100.0
)

comparison_columns = [
    "Controller",
    "K_p", "K_i", "K_d",
    "J_total", "J_tracking", "J_overshoot", "J_control",
    "overshoot_percent", "settling_time", "steady_state_error_percent",
    "voltage_max_abs", "saturation_percent", "omega_final",
    "accepted_v1", "J_improvement_vs_baseline_percent",
]

display(comparison[comparison_columns].round(6))


,Controller,K_p,K_i,K_d,J_total,J_tracking,J_overshoot,J_control,overshoot_percent,settling_time,steady_state_error_percent,voltage_max_abs,saturation_percent,omega_final,accepted_v1,J_improvement_vs_baseline_percent
0,Manual Baseline,0.800000,2.000000,0.002000,0.038177,0.000114,0.0,0.254054,0.0,1.322,0.000000,10.242803,0.0,12.600000,True,0.000000
1,Random Search,0.563947,0.233646,0.001951,0.032580,0.009169,0.0,0.180529,0.0,inf,5.029487,7.109544,0.0,12.036953,False,14.658301
2,Bayesian Optimization,0.387395,0.233312,0.001882,0.032488,0.009537,0.0,0.178437,0.0,inf,3.780578,5.820066,0.0,12.187956,False,14.901826


## Section 7 — Response Plots

같은 axes에서 speed response와 control voltage를 비교한다. Cost가 낮아졌더라도 response shape이 나빠졌는지 함께 확인해야 한다.


In [10]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

for label, result in responses.items():
    axes[0].plot(result["time"], result["omega"], label=label)
axes[0].axhline(OMEGA_REF, linestyle="--", color="black", linewidth=1, label="Reference")
axes[0].set_ylabel("omega [rad/s]")
axes[0].set_title("Speed Response Comparison")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

for label, result in responses.items():
    axes[1].plot(result["time"], result["voltage"], label=label)
axes[1].axhline(V_MAX, linestyle="--", color="tab:red", linewidth=1, label="+Vmax")
axes[1].axhline(-V_MAX, linestyle="--", color="tab:red", linewidth=1, label="-Vmax")
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("voltage [V]")
axes[1].set_title("Control Voltage Comparison")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
if SAVE_ARTIFACTS:
    plt.savefig(RESULT_FIGURE_DIR / "pid_optimization_response_comparison.png", dpi=160)
plt.show()


/var/folders/js/s5xbjpb56mn1ysk430xc81w40000gn/T/ipykernel_33176/4008823016.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(random_history["trial"], random_history["best_so_far"], marker="o", label="Random Search")
ax.plot(bayesian_history["trial"], bayesian_history["best_so_far"], marker="o", label="Bayesian Optimization")
ax.axhline(baseline_j, linestyle="--", color="black", linewidth=1, label="Manual Baseline J")
ax.set_xlabel("Evaluation trial")
ax.set_ylabel("Best J so far")
ax.set_title("Cost Convergence History")
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
if SAVE_ARTIFACTS:
    plt.savefig(RESULT_FIGURE_DIR / "pid_optimization_cost_history.png", dpi=160)
plt.show()


/var/folders/js/s5xbjpb56mn1ysk430xc81w40000gn/T/ipykernel_33176/3792969553.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 8 — Analysis

아래 해석은 실행 결과 table에서 자동으로 계산한다.

핵심 질문은 단순히 `J`가 낮아졌는지가 아니라, `J`가 낮아진 이유가 tracking 개선인지, overshoot 감소인지, control effort 감소인지 확인하는 것이다. 특히 현재 v1 cost에는 settling time과 steady-state error가 직접 term으로 들어가지 않으므로, cost improvement와 dynamic response improvement가 항상 같은 뜻은 아니다.


In [12]:
def _fmt_time(value):
    return "not settled" if not np.isfinite(value) else f"{value:.3f} s"

baseline_row = comparison.loc[comparison["Controller"] == "Manual Baseline"].iloc[0]
for controller in ["Random Search", "Bayesian Optimization"]:
    row = comparison.loc[comparison["Controller"] == controller].iloc[0]
    print(f"[{controller}]")
    print(f"J improvement vs baseline: {row['J_improvement_vs_baseline_percent']:.2f}%")
    print(f"J_total: {baseline_row['J_total']:.6f} -> {row['J_total']:.6f}")
    print(f"J_tracking: {baseline_row['J_tracking']:.6f} -> {row['J_tracking']:.6f}")
    print(f"J_control: {baseline_row['J_control']:.6f} -> {row['J_control']:.6f}")
    print(f"M_p: {baseline_row['overshoot_percent']:.3f}% -> {row['overshoot_percent']:.3f}%")
    print(f"Settling time: {_fmt_time(baseline_row['settling_time'])} -> {_fmt_time(row['settling_time'])}")
    print(f"SSE: {baseline_row['steady_state_error_percent']:.3f}% -> {row['steady_state_error_percent']:.3f}%")
    print(f"Max voltage: {baseline_row['voltage_max_abs']:.3f} V -> {row['voltage_max_abs']:.3f} V")
    print(f"Saturation: {baseline_row['saturation_percent']:.3f}% -> {row['saturation_percent']:.3f}%")
    print(f"Accepted by v1 baseline criteria: {bool(row['accepted_v1'])}")
    print()


[Random Search]
J improvement vs baseline: 14.66%
J_total: 0.038177 -> 0.032580
J_tracking: 0.000114 -> 0.009169
J_control: 0.254054 -> 0.180529
M_p: 0.000% -> 0.000%
Settling time: 1.322 s -> not settled
SSE: 0.000% -> 5.029%
Max voltage: 10.243 V -> 7.110 V
Saturation: 0.000% -> 0.000%
Accepted by v1 baseline criteria: False

[Bayesian Optimization]
J improvement vs baseline: 14.90%
J_total: 0.038177 -> 0.032488
J_tracking: 0.000114 -> 0.009537
J_control: 0.254054 -> 0.178437
M_p: 0.000% -> 0.000%
Settling time: 1.322 s -> not settled
SSE: 0.000% -> 3.781%
Max voltage: 10.243 V -> 5.820 V
Saturation: 0.000% -> 0.000%
Accepted by v1 baseline criteria: False



## Section 9 — Reproducibility

이 experiment는 다음 조건을 고정한다.

```text
random seed = 42
same nominal motor parameters
same omega_ref
same V_max
same simulation_time
same dt
same load_torque = 0
same compute_cost() weights
```

Notebook 실행 환경과 package version은 아래에 저장한다.


In [13]:
try:
    import sklearn
    sklearn_version = sklearn.__version__
except Exception:
    sklearn_version = None
try:
    import scipy
    scipy_version = scipy.__version__
except Exception:
    scipy_version = None

RUN_CONFIG = {
    "random_seed": RANDOM_SEED,
    "gain_bounds": GAIN_BOUNDS,
    "gain_bounds_source": "Temporary implementation choice from src/simulation/baseline_tuning.py scan min/max; docs/experiments/optimization_plan.md still says TBD.",
    "evaluation_budget": {
        "Random Search": N_RANDOM_TRIALS,
        "Bayesian Optimization": N_BAYESIAN_TRIALS,
        "source": "Temporary equal budget from baseline manual scan count because docs budget is TBD.",
    },
    "fixed_conditions": fixed_condition_summary.iloc[0].to_dict(),
    "cost_weights": COST_WEIGHTS,
    "acceptance": ACCEPTANCE,
    "environment": {
        **ENV,
        "matplotlib": matplotlib.__version__,
        "sklearn": sklearn_version,
        "scipy": scipy_version,
    },
}

print(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False))


{
  "random_seed": 42,
  "gain_bounds": {
    "K_p": [
      0.02,
      0.8
    ],
    "K_i": [
      0.05,
      2.0
    ],
    "K_d": [
      0.0,
      0.002
    ]
  },
  "gain_bounds_source": "Temporary implementation choice from src/simulation/baseline_tuning.py scan min/max; docs/experiments/optimization_plan.md still says TBD.",
  "evaluation_budget": {
    "Random Search": 23,
    "Bayesian Optimization": 23,
    "source": "Temporary equal budget from baseline manual scan count because docs budget is TBD."
  },
  "fixed_conditions": {
    "R": 0.18644,
    "L": 0.0063,
    "J_m": 0.013767,
    "b": 0.049813,
    "K_t": 0.020375,
    "K_e": 0.020375,
    "omega_ref_rad_s": 12.6,
    "V_max": 12.0,
    "simulation_time_s": 10.0,
    "dt_s": 0.001,
    "initial_current_A": 0.0,
    "initial_speed_rad_s": 0.0,
    "load_torque_Nm": 0.0
  },
  "cost_weights": {
    "w_tracking": 0.6,
    "w_overshoot": 0.25,
    "w_control": 0.15
  },
  "acceptance": {
    "overshoot_percent_max": 

## Section 10 — Save / Export

Optimization result는 `results/tables/`와 `results/figures/`에 저장한다.

```text
pid_optimization_comparison.csv
pid_optimization_summary.json
pid_random_search_history.csv
pid_bayesian_optimization_history.csv
pid_optimization_response_comparison.png
pid_optimization_cost_history.png
```


In [14]:
def _json_safe(value):
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, dict):
        return {k: _json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [_json_safe(v) for v in value]
    return value

summary_payload = {
    "comparison": comparison[comparison_columns].to_dict(orient="records"),
    "random_search_best": dict(random_best),
    "bayesian_optimization_best": dict(bayesian_best),
    "run_config": RUN_CONFIG,
    "motor_source": NOMINAL_MOTOR_SOURCE,
}

if SAVE_ARTIFACTS:
    comparison[comparison_columns].to_csv(RESULT_TABLE_DIR / "pid_optimization_comparison.csv", index=False)
    random_history.to_csv(RESULT_TABLE_DIR / "pid_random_search_history.csv", index=False)
    bayesian_history.to_csv(RESULT_TABLE_DIR / "pid_bayesian_optimization_history.csv", index=False)
    with open(RESULT_TABLE_DIR / "pid_optimization_summary.json", "w", encoding="utf-8") as f:
        json.dump(_json_safe(summary_payload), f, indent=2, ensure_ascii=False)

print("PID optimization records saved." if SAVE_ARTIFACTS else "SAVE_ARTIFACTS=False, no files saved.")
print(json.dumps(_json_safe({
    "best_random_search_J": random_best["total"],
    "best_bayesian_optimization_J": bayesian_best["total"],
    "baseline_J": baseline_j,
}), indent=2, ensure_ascii=False))


PID optimization records saved.
{
  "best_random_search_J": 0.032580489439631016,
  "best_bayesian_optimization_J": 0.03248752000858503,
  "baseline_J": 0.03817651842324221
}


## Final Summary

이 notebook은 `Manual Baseline`, `Random Search`, `Bayesian Optimization`을 같은 plant와 같은 simulation condition에서 비교한다.

현재 docs에서 optimization bounds와 budget이 아직 확정되지 않았으므로, 이 notebook의 bounds/budget은 final project decision이 아니라 실행 가능한 provisional setup이다. 다음 단계에서는 `docs/experiments/optimization_plan.md`의 `TBD`를 공식 값으로 확정하고, 이 notebook의 Section 3을 그 값으로 교체하면 된다.
